# Project Phase II — Student Dropout


Using the same approved problem, dataset, and project team as Phase I. The entries below are carried forward from `Phase1_Student_Dropout_Template.ipynb`.

| Field | Entry |
|---|---|
| Project title | Student Dropout Rate |
| Group members | Anas — 24253881 · Hasan — 24245408 · Aaron Taylor — 24232594 |
| Dataset | Predict Students' Dropout and Academic Success (`data/data.csv`, UCI ML Repository) |
| Target variable | `Target` — Dropout, Enrolled, Graduate |
| Phase | Project Phase II |


Work Plan: tick when stage complete

- Task 1: All
- Task 2: Aaron
- Task 3: Hasan
- Task 4: Anas
- Task 5: Aaron
- Task 6: Hasan
- Task 7: Anas
- Report: All


## Setup


- Import the libraries and load the dataset used in Phase I.

In [16]:
import importlib.util
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}
missing = [pip_name for mod, pip_name in required_packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "data/data.csv"
df_raw = pd.read_csv(DATA_PATH, sep=";")  # same semicolon-separated file as Phase I

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("Shape:", df_raw.shape)
print("Random seed:", RANDOM_SEED)
print(df_raw["Target"].value_counts())


Python: 3.14.6
pandas: 3.0.6
Shape: (4424, 37)
Random seed: 42
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64


# Task 1: Project Phase I Summary

What to write:

- Restate the selected problem, dataset, target variable, and prediction objective.
- Summarise the dataset source, number of records, number of features, and main data characteristics.
- Briefly describe the main data-quality issues identified in Phase I.
- Explain why the problem is suitable for an LCS-based solution.

Facts already established in Phase I, for reuse in the write-up:

| Item | Phase I result |
|---|---|
| Problem | Identify students at risk of dropping out from academic, demographic, and socio-economic factors, so support can be offered earlier. |
| Prediction objective | Predict student academic outcome, with dropout risk as the decision the model is meant to support. |
| Source | UC Irvine ML Repository, Realinho et al. (2022). Local file: `data/data.csv`. |
| Size | 4,424 records, 36 predictors + `Target`. |
| Target | Three classes: Graduate (about 49.9%), Dropout (about 32.1%), Enrolled (about 17.9%). |
| Main quality notes | No missing values and no duplicate rows in the published file. Age at enrollment is right-skewed. Several integer columns are category codes, not true continuous measures. Class imbalance is moderate and should be handled explicitly. |
| Modelling caveat from Phase I | 2nd-semester academic variables are only available part-way through the year. An early-warning use case may need a feature set limited to information known at enrollment. |


*Write the Task 1 summary here.*


# Task 2: Original LCS System on the Raw Dataset

What to do:

- Select an LCS variant. eLCS is the recommended supervised variant.
- Use the original LCS code with no algorithmic modification.
- Run it on the raw or minimally processed dataset.
- Report the selected LCS parameters.
- Record the baseline performance.
- Explain any minimal processing required only to make the dataset compatible with the LCS code.

Keep this run separate from Task 4. Do not tune, prune, or redesign the algorithm here. The only allowed changes are those required for the data to run in the chosen implementation (for example, a numeric class label, or a train/test split the code expects).


In [17]:
# Task 2 — unmodified eLCS on the raw / minimally processed dataset.
# Uses df_raw and RANDOM_SEED from Setup. Fit transforms only happen here for label encoding.

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("skeLCS") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-elcs"])

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from skeLCS import eLCS

CLASS_NAMES = {0: "Dropout", 1: "Enrolled", 2: "Graduate"}
TARGET_MAP = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
TEST_SIZE = 0.20

# Minimal processing only: keep all rows and columns; make the frame eLCS-compatible.
raw_frame = df_raw.copy()
raw_frame.columns = [str(c).strip() for c in raw_frame.columns]
assert raw_frame["Target"].isna().sum() == 0

y_raw = raw_frame["Target"].map(TARGET_MAP).to_numpy()
feature_names = [c for c in raw_frame.columns if c != "Target"]
X_raw = raw_frame[feature_names].to_numpy(dtype=float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y_raw,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y_raw,
)

print("Train shape:", X_train_raw.shape, "Test shape:", X_test_raw.shape)
print("Train class counts:\n", pd.Series(y_train).map(CLASS_NAMES).value_counts())
print("Test class counts:\n", pd.Series(y_test).map(CLASS_NAMES).value_counts())

# Unmodified eLCS: library defaults, plus random_state only for reproducibility.
lcs_original = eLCS(random_state=RANDOM_SEED)
print("\nSelected parameters (library defaults + random_state):")
for name in sorted(lcs_original.get_params()):
    print(f"  {name}: {lcs_original.get_params()[name]}")

lcs_original.fit(X_train_raw, y_train)
y_pred_raw = lcs_original.predict(X_test_raw)

acc = accuracy_score(y_test, y_pred_raw)
bal = balanced_accuracy_score(y_test, y_pred_raw)
macro_f1 = f1_score(y_test, y_pred_raw, average="macro")
majority = (y_test == 2).mean()  # Graduate is the majority class in this dataset
cm = confusion_matrix(y_test, y_pred_raw, labels=[0, 1, 2])

print("\nBaseline results on stratified 80/20 test set:")
print(f"  Accuracy:           {acc:.4f}")
print(f"  Majority baseline:  {majority:.4f} (always predict Graduate)")
print(f"  Balanced accuracy:  {bal:.4f}")
print(f"  Macro F1:           {macro_f1:.4f}")
print("\nConfusion matrix (rows=actual, cols=predicted; Dropout, Enrolled, Graduate):")
print(cm)
print("\nClassification report:")
print(classification_report(y_test, y_pred_raw, target_names=["Dropout", "Enrolled", "Graduate"]))


Train shape: (3539, 36) Test shape: (885, 36)
Train class counts:
 Graduate    1767
Dropout     1137
Enrolled     635
Name: count, dtype: int64
Test class counts:
 Graduate    442
Dropout     284
Enrolled    159
Name: count, dtype: int64

Selected parameters (library defaults + random_state):
  N: 1000
  acc_sub: 0.99
  beta: 0.2
  chi: 0.8
  delta: 0.1
  discrete_attribute_limit: 10
  do_GA_subsumption: True
  do_correct_set_subsumption: False
  fitness_reduction: 0.1
  init_fit: 0.01
  learning_iterations: 10000
  match_for_missingness: False
  mu: 0.04
  nu: 5
  p_spec: 0.5
  random_state: 42
  reboot_filename: None
  selection_method: tournament
  specified_attributes: []
  theta_GA: 25
  theta_del: 20
  theta_sel: 0.5
  theta_sub: 20
  track_accuracy_while_fit: False

Baseline results on stratified 80/20 test set:
  Accuracy:           0.7243
  Majority baseline:  0.4994 (always predict Graduate)
  Balanced accuracy:  0.5867
  Macro F1:           0.5460

Confusion matrix (rows=act

*Task 2 write-up: variant, parameters, minimal processing, baseline results.*


# Task 3: Data Preprocessing and Feature Engineering

What to do:

- Address missing values, duplicate records, invalid values, inconsistent categories, and incorrect data types.
- Investigate and handle outliers where appropriate.
- Encode categorical variables and scale, discretise, or transform numerical variables where required.
- Address class imbalance where relevant.
- Apply feature engineering, feature selection, or dimensionality reduction where justified.
- Explain how the preprocessing is expected to improve LCS performance.
- Keep preprocessing free of data leakage. Fit any transform on the training data only, then apply it to the test data.

What to produce: the cleaned and feature-engineered dataset, or clear instructions that reproduce it.

Phase I already found no missing values and no duplicate rows, and it created both a raw and a capped age column plus some scaled grades. Revisit those choices here for the LCS, rather than assuming every Phase I transform should be reused. Integer category codes (course, occupation, qualification, and similar fields) still need an encoding decision. Class imbalance (Graduate / Dropout / Enrolled) still needs an explicit handling decision.

Save the modelling-ready data, or document the exact steps in the code cell so a marker can reproduce it from `data/data.csv`.


In [ ]:
import numpy as np
import pandas as pd


df_cleaned = df_raw.copy()

print("Initial raw dataset shape:", df_raw.shape)

# ==============================================================================
# 1. CORRECT FORMAT & COLUMN HEADERS
# ==============================================================================

# Strip whitespace and clean special characters from column names
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.replace(" ", "_")
    .str.replace("'", "")
    .str.replace("/", "_")
)

# Standardize text strings across all text columns (strip whitespace, consistent casing)
text_cols = df_cleaned.select_dtypes(include=["object"]).columns
for col in text_cols:
  df_cleaned[col] = df_cleaned[col].astype(str).str.strip()

# Replace missing value placeholder strings/numbers with actual np.nan
df_cleaned = df_cleaned.replace(
    ["?", "NaN", "nan", "null", "NULL", "", " ", 999, -99, -999], np.nan
)


# ==============================================================================
# 2. ENCODE TARGET VARIABLE TO NUMERIC
# ==============================================================================

# Map categorical target string to numerical values: Dropout -> 0, Enrolled -> 1, Graduate -> 2
if "Target" in df_cleaned.columns:
  target_mapping = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
  df_cleaned["Target"] = df_cleaned["Target"].map(target_mapping)


# ==============================================================================
# 3. CORRECT CATEGORIES & FIX INCONSISTENT VALUES
# ==============================================================================

# Clean and standardize binary/categorical labels (e.g. standardizing gender if text)
if "Gender" in df_cleaned.columns:
  gender_map = {
      "Male": 1,
      "Female": 0,
      "M": 1,
      "F": 0,
      "1": 1,
      "0": 0,
      1: 1,
      0: 0,
  }
  df_cleaned["Gender"] = df_cleaned["Gender"].map(gender_map)

# Fix invalid numerical boundaries (e.g., negative grades or grades > 20 in Portuguese scale)
grade_cols = [c for c in df_cleaned.columns if "grade" in c.lower()]
for col in grade_cols:
  df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce")
  df_cleaned.loc[(df_cleaned[col] < 0) | (df_cleaned[col] > 20), col] = np.nan


# ==============================================================================
# 4. ADDRESS OUTLIERS (IQR Capping / Winsorization)
# ==============================================================================

# Define continuous numerical features where extreme outliers should be capped
continuous_cols = [
    c
    for c in [
        "Age_at_enrollment",
        "Curricular_units_1st_sem_grade",
        "Curricular_units_2nd_sem_grade",
        "Unemployment_rate",
        "Inflation_rate",
        "GDP",
    ]
    if c in df_cleaned.columns
]

for col in continuous_cols:
  df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce")
  Q1 = df_cleaned[col].quantile(0.25)
  Q3 = df_cleaned[col].quantile(0.75)
  IQR = Q3 - Q1
  lower_bound = Q1 - 1.5 * IQR
  upper_bound = Q3 + 1.5 * IQR
  # Cap extreme outliers to upper and lower IQR boundaries
  df_cleaned[col] = np.clip(df_cleaned[col], lower_bound, upper_bound)


# ==============================================================================
# 5. ADDRESS MISSING VALUES (IMPUTATION)
# ==============================================================================

# Identify categorical columns vs numerical columns
categorical_cols = [
    c
    for c in [
        "Marital_status",
        "Application_mode",
        "Course",
        "Daytime_evening_attendance",
        "Previous_qualification",
        "Mother_qualification",
        "Father_qualification",
        "Mother_occupation",
        "Father_occupation",
        "Displaced",
        "Educational_special_needs",
        "Debtor",
        "Tuition_fees_up_to_date",
        "Gender",
        "Scholarship_holder",
        "International",
    ]
    if c in df_cleaned.columns
]

numerical_cols = [
    c
    for c in df_cleaned.columns
    if c not in categorical_cols + ["Target"] and c not in text_cols
]

# Impute missing numerical values using Median
for col in numerical_cols:
  df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors="coerce")
  if df_cleaned[col].isnull().sum() > 0:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].median())

# Impute missing categorical values using Mode
for col in categorical_cols:
  if df_cleaned[col].isnull().sum() > 0:
    df_cleaned[col] = df_cleaned[col].fillna(df_cleaned[col].mode()[0])


# ==============================================================================
# 6. APPLY CATEGORICAL FEATURE ENCODING
# ==============================================================================

# One-Hot Encode multi-class categorical variables (drop_first=True to avoid redundancy)
df_cleaned = pd.get_dummies(
    df_cleaned,
    columns=[c for c in categorical_cols if c in df_cleaned.columns],
    drop_first=True,
    dtype=int,
)

print("\nData cleaning complete!")
print("Final processed dataset shape:", df_cleaned.shape)
print("Target value counts:\n", df_cleaned["Target"].value_counts(dropna=False))

*Explain each preprocessing decision, and how it is expected to improve LCS performance. State where the cleaned dataset is saved, or how to reproduce it from this notebook.*


# Task 4: Improved LCS-Based System

What to do:

- Improve the LCS code, the LCS configuration, or the modelling pipeline. Possible changes include hyperparameter tuning, preprocessing designed for LCS, feature selection or dimensionality reduction, rule pruning or rule compaction, a modified LCS algorithm, class-imbalance handling, or cost-sensitive learning.
- State clearly what changed compared with the original LCS.
- Justify why the change may improve prediction accuracy or model quality.
- Provide a brief algorithmic explanation, pseudocode, or flowchart of the improved system.
- Compare the improved system with the original LCS baseline.

One clear, justified change is enough if the group can explain it. Record the change before interpreting the scores, so the comparison in Task 6 has a defined "before" and "after".


In [19]:
# Task 4 — improved LCS.
# Keep the Task 2 implementation unchanged.
# Put the modification, tuned configuration, or extra pipeline step in this cell.


*State what changed relative to the original LCS, why that change may improve accuracy or model quality, and give a short algorithmic explanation, pseudocode, or flowchart.*

| Item | Original LCS (Task 2) | Improved LCS (this task) |
|---|---|---|
| Code or configuration that changed | | |
| Data the model sees | Raw or minimally processed | Preprocessed / feature-engineered |
| Why the change may help | | |


# Task 5: Experiments

What to do for validation:

- Use either an 80/20 train-test split or k-fold cross-validation.
- Justify the chosen validation approach.

What to do for metrics. This is a classification problem, so consider:

- Accuracy
- Precision
- Recall
- F1-score
- Balanced accuracy
- ROC-AUC
- PR-AUC
- Confusion matrix

Select the metrics that fit this problem and justify that selection.

The target has three classes and moderate imbalance (Graduate is about half the rows; Enrolled is the smallest class). Accuracy alone can look strong while missing Dropout or Enrolled. Prefer metrics that show performance on each class, and say which class matters most for an early-support decision.

What to do for statistical testing:

- Apply at least one suitable statistical test to compare model performance.
- Interpret whether the performance differences are statistically significant.

Use the same validation procedure for every model in Task 6.


In [20]:
# Task 5 — validation, metrics, and a statistical comparison.
# Define the split or folds once, with RANDOM_SEED, and reuse that procedure in Task 6.


*Justify the validation approach, name the selected metrics and why they fit this three-class problem, and interpret the statistical test.*

| Choice | Decision | Justification |
|---|---|---|
| Validation | | |
| Metrics | | |
| Statistical test | | |
| Significant difference? | | |


# Task 6: Model Comparison

Compare all of the following on the same validation setup:

1. Original LCS on the raw or minimally processed dataset.
2. Original LCS on the preprocessed dataset.
3. Improved LCS on the preprocessed and/or feature-engineered dataset.
4. At least three conventional non-deep-learning models.

Conventional models may include Random Forest, Support Vector Machine, Decision Tree, Logistic Regression, Naive Bayes, k-Nearest Neighbours, or Gaussian Process. Deep-learning models are not permitted for this comparison.

Record which three conventional models were used:

| Model | Included |
|---|---|
| 1 | |
| 2 | |
| 3 | |

Put the comparison results in this table after the models have been run.

| Model | Data used | Accuracy | Balanced accuracy | Macro F1 | Other chosen metrics |
|---|---|---|---|---|---|
| Original LCS | Raw or minimally processed | | | | |
| Original LCS | Preprocessed | | | | |
| Improved LCS | Preprocessed / feature-engineered | | | | |
| Conventional model 1 | | | | | |
| Conventional model 2 | | | | | |
| Conventional model 3 | | | | | |


In [21]:
# Task 6 — same validation setup for every row of the comparison table.
# 1. Original LCS, raw or minimally processed data (reuse Task 2 if the split matches).
# 2. Original LCS, preprocessed data.
# 3. Improved LCS, preprocessed and/or feature-engineered data.
# 4. Three conventional models. No deep-learning models.


*Fill this table from the Task 6 run. Add the extra metric columns chosen in Task 5.*

| Model | Data used | Accuracy | Balanced accuracy | Macro F1 | Other chosen metrics |
|---|---|---|---|---|---|
| Original LCS | Raw or minimally processed | | | | |
| Original LCS | Preprocessed | | | | |
| Improved LCS | Preprocessed / feature-engineered | | | | |
| Conventional model 1 | | | | | |
| Conventional model 2 | | | | | |
| Conventional model 3 | | | | | |


# Task 7: Interpretation of Results

What to write, in a separate interpretation section:

- Interpret the main findings from the model comparison.
- Explain important LCS rules, feature conditions, or classifier patterns.
- Use at least three sample LCS rules or classifiers to discuss the decision-making process.
- Discuss how the LCS rule-based outputs support interpretability.
- Discuss whether the explanations are trustworthy and practically meaningful.

Record three rules here after they are exported from the LCS.

| Rule | Conditions | Predicted class | What the group will explain |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |


In [22]:
# Task 7 — export rules or classifiers from the improved LCS (or the original LCS if that is the system being interpreted).
# Copy three of them into the table below.


*Write the interpretation here. Cover the comparison findings, three sample rules, interpretability, and whether those explanations are trustworthy in a student-support setting.*

| Rule | Conditions | Predicted class | What the group will explain |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |


# Task 8: Discussion, Limitations, and Future Work

What to discuss:

- Reasons for the performance improvement, or for the lack of improvement.
- The practical meaning of the results, not only the numerical scores.
- Limitations from data quality, dataset size, class imbalance, noise, missingness, or feature relevance.
- Future work, such as another LCS variant, better preprocessing, external validation, or review by a domain expert.
- Responsible use of the system.

Points already visible from Phase I that this discussion should not ignore:

- The data come from one Portuguese higher-education institution.
- Outcome labels are assigned at the end of the normal course duration, so they are not an in-semester alert by themselves.
- Semester performance features, especially 2nd-semester approved units, are strong but late. A model that depends on them cannot support intervention at enrollment.
- Class imbalance can make accuracy misleading.
- A rule that flags a student is not, by itself, a reason to withhold support or to make a high-stakes decision without a person reviewing it.


*Write the discussion, limitations, future work, and responsible-use notes here.*


# Contribution Statement

Each member writes a short individual reflection: what they built, what they checked, and what they can explain in the demonstration.

| Member | Individual reflection |
|---|---|
| Anas — 24253881 | |
| Hasan — 24245408 | |
| Aaron Taylor — 24232594 | |


*Each member replaces the placeholder in their row. Write in the first person.*

| Member | Individual reflection |
|---|---|
| Anas — 24253881 | |
| Hasan — 24245408 | |
| Aaron Taylor — 24232594 | |


# Deliverables checklist

| Deliverable | Done |
|---|---|
| Final report, 5,000–6,500 words, excluding references and appendices | |
| Source code or Jupyter notebook | |
| GitHub repository link and commit-history evidence | |
| Contribution statement, with an individual reflection from each member | |
| Cleaned and feature-engineered dataset, or clear reproduction instructions | |
| Turnitin similarity report at or below 20% | |

Repository link:

| Item | Entry |
|---|---|
| GitHub repository URL | |
| Commit-history evidence (link to the commits page, or a short note on where it is shown) | |
